# 07. Source-balanced training weights

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.transfer import transfer
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
cohorts = list(transfer.FULL_COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)

## Base configurations

In [ ]:
augmentation_run = ""  # Optional pipeline_outputs/transfer/augmentation_<id>; blank uses the latest
augmentation_run = Path(augmentation_run) if augmentation_run else tuning.latest_run(root, "augmentation", area=transfer.AREA)
choice = json.loads((augmentation_run / "selected.json").read_text())
print("augmentation run:", augmentation_run)
choice

params = {cohort: {**choice[cohort]["params"], "n_estimators": 300} for cohort in cohorts}
bases = {cohort: {"plain": {"variant": "interval_72", "augmentation": None},
                  "best_so_far": {"variant": choice[cohort]["series_variant"],
                                  "augmentation": choice[cohort]["augmentation_settings"]}} for cohort in cohorts}
pd.DataFrame([{"cohort": c, "base": b, **{k: str(v) for k, v in s.items()}} for c in cohorts for b, s in bases[c].items()])

## Options

In [ ]:
options = {"crop_parcel": {"weighting": "crop_parcel"},
           "crop_source_parcel": {"weighting": "crop_source_parcel"}}
options

## Weight shares by crop and source

In [ ]:
shares = []
for cohort in cohorts:
    train, _ = tuning.cohort_split(development, cohort)
    for weighting in transfer.WEIGHTINGS:
        weights = transfer.training_weights(train, weighting)
        table = pd.DataFrame({"crop": [r["crop"] for r in train], "source": [r["source_dataset"] for r in train],
                              "weight": weights}).groupby(["crop", "source"]).weight.sum() / weights.sum()
        shares.extend({"cohort": cohort, "weighting": weighting, "crop": crop, "source": source, "share": share}
                      for (crop, source), share in table.items())
display(pd.DataFrame(shares).pivot_table(index=["cohort", "crop", "source"], columns="weighting", values="share").round(3))

## Evaluate

In [ ]:
run = tuning.new_run(root, "source_weights", features_path, area=transfer.AREA, augmentation_run=str(augmentation_run),
                     options={k: str(v) for k, v in options.items()})
summaries, source_details, crop_details = [], [], []
for cohort in cohorts:
    for base, setting in bases[cohort].items():
        for name, training in options.items():
            summary, details = transfer.evaluate_configuration(
                development, cohort, params[cohort], setting["variant"], setting["augmentation"],
                training, label=f"{base} | {name}")
            summaries.append({"base": base, "option": name, **summary})
            source_details.extend({"base": base, "option": name, **r} for r in details["source"])
            crop_details.extend({"base": base, "option": name, **r} for r in details["crop"])
            print(cohort, base, name, summary.get("status"), round(summary.get("spatial_macro_f1") or 0, 3), flush=True)
tuning.save_records(run, "summary", summaries)
tuning.save_records(run, "source_holdout", source_details)
tuning.save_records(run, "crop_holdout", crop_details)

## Summary table

In [ ]:
summary = pd.DataFrame(summaries)
columns = ["cohort", "base", "option", "status", "spatial_macro_f1", "fold_min", "locked_validation_macro_f1",
           "source_holdout_macro_f1_mean", "crop_holdout_recall_mean", "spatial_carrot_f1", "spatial_cabbage_f1", "spatial_potato_f1"]
display(summary[[c for c in columns if c in summary]].round(3))

## Leave-one-source-out and per-crop holdout details

In [ ]:
display(pd.DataFrame(source_details).round(3))
display(pd.DataFrame(crop_details).round(3))

## Figure

In [ ]:
scored = summary[summary.status == "scored"]
metrics = ["spatial_macro_f1", "locked_validation_macro_f1", "source_holdout_macro_f1_mean"]
figure, axes = plt.subplots(len(cohorts), 2, figsize=(14, 4 * len(cohorts)), squeeze=False)
for i, cohort in enumerate(cohorts):
    for j, base in enumerate(["plain", "best_so_far"]):
        table = scored[(scored.cohort == cohort) & (scored.base == base)].set_index("option")[metrics]
        if len(table):
            table.plot(kind="bar", ax=axes[i][j], legend=(i == 0 and j == 1))
        axes[i][j].set_title(f"{cohort} — {base}")
        axes[i][j].set_ylim(0, 1)
show_and_save(figure, run, "source_weights")
print("Run folder:", run)